# Structured streaming Demo

In [1]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime




In [2]:
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0") \
    .getOrCreate()

In [3]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [4]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

### Enriching the Streaming Analysis

For this step, we added some new aggregations to our PySpark Streaming code.

First, we changed the time window from 1 hour to 30 minutes. This allows us to analyse the Wikipedia edits in more detail and see what happens during shorter periods.

Then, we looked at the size of the edits made to Wikipedia pages. We calculated the edit size by comparing the new size of the page with its old size:
edit_size = length.new - length.old

We also added some aggregations to get more information about the edits like :
edit_count gives the number of edits.
avg_edit_size gives the average size of the edits.
max_edit_size gives the size of the biggest edit.
total_edit_size gives the total size of all the edits.
The results are grouped by the French Wikipedia (frwiki) and by a 30-minute time window.
This gives us more information than just counting the edits because we can also see how much the Wikipedia pages were modified.

In [5]:
from pyspark.sql.functions import avg, sum, max, min

edit_size_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .withColumn(
        "edit_size",
        col("length.new") - col("length.old")
    ) \
    .groupBy(
        window(col("event_time"), "30 minutes"),
        col("wiki")
    ) \
    .agg(
        count("*").alias("edit_count"),
        avg("edit_size").alias("avg_edit_size"),
        max("edit_size").alias("max_edit_size"),
        sum("edit_size").alias("total_edit_size")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("wiki"),
        col("edit_count"),
        col("avg_edit_size"),
        col("max_edit_size"),
        col("total_edit_size")
    )

The results show the activity on the French Wikipedia (frwiki) during different time windows.

For the first analysis, we can see the number of edits made by bots and non-bots for each one-hour window. For example, between 10:00 and 11:00, there were 768 edits from non-bot users and 169 edits from bots.

For the second analysis, we used 30-minute windows and looked at the size of the edits. For example, between 15:00 and 15:30, there were 369 edits, with an average edit size of about 325. The biggest edit was 15,757, and the total edit size was 119,989.

These results show that we can use PySpark Streaming not only to count the number of edits, but also to analyse the size and activity of the changes made to Wikipedia pages.

In [6]:
edit_size_query = edit_size_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

In [7]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

In [8]:
# Use console output (goes to Docker logs)
streaming_query_df = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

In [9]:
edit_size_query.stop()
streaming_query_df.stop()

### Tumbling and Overlapping Windows

For this step, we created two different types of windows.

First, we used a tumbling window of 30 minutes. Each window lasts 30 minutes and does not overlap with the next one.

Then, we created an overlapping window of 30 minutes with a sliding interval of 15 minutes. This means that a new window starts every 15 minutes, so two consecutive windows overlap for 15 minutes.
    

In [10]:
# Overlapping window: 30 minutes window, every 15 minutes

overlapping_window_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "30 minutes", "15 minutes"),
        col("wiki")
    ) \
    .agg(
        count("*").alias("edit_count")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("wiki"),
        col("edit_count")
    )

In [11]:
overlapping_query = overlapping_window_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

The results show the number of edits made on Wikipedia during each 30-minute window. Since the window starts every 15 minutes, the windows overlap by 15 minutes.

For example, the windows 09:15–09:45 and 09:30–10:00 overlap. The edit_count column shows the number of edits during each window.

We can also see both frwiki and enwiki in the results because the Kafka topic contains events from both Wikipedia languages.
These results confirm that the overlapping window works correctly.